# Trendline-walk scan — honest fill + MAXSPAN sweep

Runs every symbol through the corrected detector and reports per-symbol metrics.

- `old(open,causalF)` = the pre-correction settings (break-through fill, refit lookahead ON). Kept only for comparison — these numbers are inflated.
- `intrabar sNNNN` = the honest model (touch-fill resting stop, causal pivot confirmation ON) at each MAXSPAN listed.

**Setup**
1. Put the updated `backtest_hull.py` in the same folder as this notebook.
2. Set `DATA_DIR` below to the folder holding the data files. FirstRateData `.txt` files work as-is.
3. `pip3 install numpy pandas` if needed.

**Runtime**: roughly 3-6s per symbol per config. Start with the 5-symbol test (`SYMBOLS` preset below) — it should reproduce GC PF 1.21 at span 1200, which confirms the config is right before committing to the full run.

In [ ]:
import os, glob, re, time
import numpy as np
import pandas as pd
from backtest_hull import run, load_series

# ---------------- config ----------------
DATA_DIR    = "DATA_DIR"      # folder with the data files (.txt FirstRateData or .csv)
WARMUP_DAYS = 60              # history before first entry (warms 200 EMA / ATR)
MAXSPANS    = [1200]          # max trendline age in bars. [1200] matches the CLI run.
                              # add 10**9 for an unbounded sweep: [1200, 10**9]
RUN_OLD     = True            # also run the pre-correction settings for comparison
SLIP_TICKS  = 0               # round-trip slippage in ticks; 0 = off. only hits symbols in TICKS
SYMBOLS     = ["GC","PL","HG","PA","SI"]   # quick metals test. Set to None for all symbols.

TICKS = {"MGC":0.10,"GC":0.10,"SI":0.005,"SIL":0.005,"HG":0.0005,"MHG":0.0005,
         "PL":0.10,"PA":0.05,"CL":0.01,"MCL":0.01,"NQ":0.25,"MNQ":0.25,"ES":0.25,"MES":0.25}

print(f"DATA_DIR={DATA_DIR!r}  MAXSPANS={MAXSPANS}  RUN_OLD={RUN_OLD}  SLIP_TICKS={SLIP_TICKS}")
print("SYMBOLS =", "ALL" if SYMBOLS is None else SYMBOLS)

## Discover data files

In [ ]:
def symbol_of(path):
    b = os.path.basename(path)
    b = re.sub(r"_full_1hour.*$", "", b)     # A6_full_1hour_continuous_ratio_adjusted.txt -> A6
    b = re.sub(r"\.(csv|txt)$", "", b)       # plain SYM.csv / SYM.txt
    b = re.sub(r"^\d+_", "", b)              # 1782_MGC -> MGC
    return b

paths = sorted(glob.glob(os.path.join(DATA_DIR, "*.txt")) + glob.glob(os.path.join(DATA_DIR, "*.csv")))
catalog = {symbol_of(p): p for p in paths}
assert catalog, (f"no .txt or .csv files found in {DATA_DIR}/ -- check DATA_DIR, and make sure "
                 "the files are downloaded locally (no iCloud cloud icons in Finder)")

todo = sorted(catalog) if SYMBOLS is None else [s for s in SYMBOLS if s in catalog]
missing = [] if SYMBOLS is None else [s for s in SYMBOLS if s not in catalog]

print(f"found {len(catalog)} symbols in {DATA_DIR}/  |  running {len(todo)}")
if missing:
    print("NOT FOUND (check the symbol names):", ", ".join(missing))
print(", ".join(todo))

## Run the scan

In [ ]:
KEEP = ("trades","win","totalR","avgR","PF","long","short")

def metric(sym, path, maxspan, fill, causal):
    T,*_ = load_series(path)
    start = int(T.min()) + WARMUP_DAYS*86400
    s,_,_,_ = run(sym, path, 1.0, start, CAP=1e12, MAXSPAN=maxspan,
                  fill=fill, causal=causal, tick=TICKS.get(sym,0.0), slip_ticks=SLIP_TICKS)
    return s

rows, t0 = [], time.time()
for i, sym in enumerate(todo, 1):
    path = catalog[sym]
    try:
        if RUN_OLD:
            s = metric(sym, path, 1200, "open", False)
            rows.append(dict(symbol=sym, config="old(open,causalF)", maxspan=1200, **{k:s[k] for k in KEEP}))
        for ms in MAXSPANS:
            s = metric(sym, path, ms, "intrabar", True)
            rows.append(dict(symbol=sym, config=f"intrabar s{ms}", maxspan=ms, **{k:s[k] for k in KEEP}))
        last = rows[-1]
        print(f"  [{i}/{len(todo)}] {sym:6s} {last['trades']:5d}tr  PF {last['PF']}  "
              f"{last['totalR']:+.1f}R   ({time.time()-t0:.0f}s)", flush=True)
    except Exception as e:
        rows.append(dict(symbol=sym, config="ERROR", maxspan=None, note=f"{type(e).__name__}: {e}"[:80]))
        print(f"  [{i}/{len(todo)}] {sym:6s} ERROR {type(e).__name__}", flush=True)

df = pd.DataFrame(rows)
print(f"\ndone in {time.time()-t0:.0f}s, {len(df)} rows")

## Sanity check
GC at span 1200 honest should be PF 1.21, 974 trades, +81.6R. If it is not, the config is wrong — stop and fix it before reading anything below.

In [ ]:
chk = df[(df.symbol=="GC") & (df.config=="intrabar s1200")]
if len(chk):
    r = chk.iloc[0]
    ok = (abs(r.PF - 1.21) < 0.02) and (abs(r.totalR - 81.6) < 1.0)
    print(f"GC honest @1200: {int(r.trades)} trades, PF {r.PF}, {r.totalR:+.1f}R")
    print("MATCHES the CLI run" if ok else "*** DOES NOT MATCH — check DATA_DIR / MAXSPANS / SLIP_TICKS ***")
else:
    print("GC @ span 1200 not in this run — skipping the check")

## Per-symbol table (honest settings)

In [ ]:
_ib = sorted([c for c in df.config.unique() if str(c).startswith("intrabar s")],
             key=lambda c: int(c.rsplit("s", 1)[-1]))
assert _ib, "no honest (intrabar) rows in df -- did the scan cell error out?"
PRIMARY_CFG = _ib[-1]

def _lbl(c):
    if str(c).startswith("intrabar s"):
        n = int(c.rsplit("s", 1)[-1])
        return "intrabar (max span)" if n >= 10**7 else f"intrabar {n}"
    return c

print("displaying:", PRIMARY_CFG, "->", _lbl(PRIMARY_CFG))

pd.set_option("display.max_rows", None)
pd.set_option("display.width", 220)

view = (df[df.config == PRIMARY_CFG]
        [["symbol","trades","win","totalR","avgR","PF","long","short"]]
        .sort_values("totalR", ascending=False)
        .reset_index(drop=True)
        .rename(columns={"win":"win%"}))
view

## What the correction cost — old vs honest

This is the important table. Each row shows the same symbol under the pre-correction settings and under the honest ones. The gap is what the two look-ahead bugs were worth.

In [ ]:
if RUN_OLD:
    comp = df[df.config.isin(["old(open,causalF)", PRIMARY_CFG])].pivot_table(
        index="symbol", columns="config", values=["PF","totalR","trades"], aggfunc="first")
    comp.columns = [f"{a}_{'old' if 'old' in b else 'honest'}" for a,b in comp.columns]
    comp["PF_delta"]     = (comp["PF_honest"] - comp["PF_old"]).round(2)
    comp["totalR_delta"] = (comp["totalR_honest"] - comp["totalR_old"]).round(1)
    comp = comp[["trades_old","trades_honest","PF_old","PF_honest","PF_delta",
                 "totalR_old","totalR_honest","totalR_delta"]].sort_values("PF_honest", ascending=False)
    display(comp.round(2))
else:
    print("RUN_OLD is False -- set it to True to get this comparison")

## MAXSPAN sweep — total R per symbol across spans

In [ ]:
cols = (["old(open,causalF)"] if RUN_OLD else []) + _ib
pivR = df[df.config != "ERROR"].pivot_table(index="symbol", columns="config", values="totalR", aggfunc="first")
pivR = pivR[[c for c in cols if c in pivR.columns]].sort_values(PRIMARY_CFG, ascending=False)
pivR.rename(columns=_lbl).round(1)

## MAXSPAN sweep — profit factor per symbol across spans

In [ ]:
pivPF = df[df.config != "ERROR"].pivot_table(index="symbol", columns="config", values="PF", aggfunc="first")
pivPF = pivPF[[c for c in cols if c in pivPF.columns]].sort_values(PRIMARY_CFG, ascending=False)
pivPF.rename(columns=_lbl).round(2)

## Portfolio totals and survivor counts

How many symbols stay net positive under each setting — the answer to "do I still have a book". The honest survivor count is the number that belongs on a resume, not the old one.

In [ ]:
g = df[df.config != "ERROR"].copy()
summary = (g.groupby("config")
             .agg(symbols=("symbol","nunique"),
                  trades=("trades","sum"),
                  totalR=("totalR","sum"),
                  mean_PF=("PF","mean"))
             .reindex([c for c in cols if c in g.config.unique()]))
summary.index = [_lbl(c) for c in summary.index]
print(summary.round(2))
print()
for cfg in cols:
    sub = g[g.config == cfg]
    if len(sub):
        pos = int((sub.totalR > 0).sum())
        print(f"{_lbl(cfg):22s}: {pos}/{sub.symbol.nunique()} symbols net positive, total {sub.totalR.sum():+.0f}R")

errs = df[df.config == "ERROR"]
if len(errs):
    show = [c for c in ["symbol","note"] if c in errs.columns]
    print("\nerrors:"); print(errs[show].to_string(index=False))

## Save results

In [ ]:
stamp = time.strftime("%Y%m%d")
df.to_csv(f"scan_results_long_{stamp}.csv", index=False)
pivR.rename(columns=_lbl).to_csv(f"scan_maxspan_totalR_{stamp}.csv")
pivPF.rename(columns=_lbl).to_csv(f"scan_maxspan_PF_{stamp}.csv")
out = [f"scan_results_long_{stamp}.csv", f"scan_maxspan_totalR_{stamp}.csv", f"scan_maxspan_PF_{stamp}.csv"]
if RUN_OLD:
    comp.round(2).to_csv(f"scan_old_vs_honest_{stamp}.csv"); out.append(f"scan_old_vs_honest_{stamp}.csv")
print("saved:", ", ".join(out))